In [1]:
import os
os.chdir("d:/Kidney_classification_Using_MLOPS_and_DVC_Data-version-control")
print("Working directory:", os.getcwd())

Working directory: d:\Kidney_classification_Using_MLOPS_and_DVC_Data-version-control


In [2]:
import os
import dagshub

os.environ["MLFLOW_TRACKING_USERNAME"] = "sentongo-web"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "9c33e4038348a07b160fd4a22d5612be534eaa10"

dagshub.init(
    repo_owner="sentongo-web",
    repo_name="Kidney_classification_Using_MLOPS_and_DVC_Data-version-control",
    mlflow=True
)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\Users\sento\anaconda3\envs\kidney\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=b1dd7906-2f2f-46f7-9c70-8777180185cd&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=5c8aa470bc3d443800fb171bba6dc5dfe1747876f0dfba7783b96a9309b0c337




Accessing as sentongo-web

Repository Kidney_classification_Using_MLOPS_and_DVC_Data-version-control doesn't exist, creating it under current 
user.

Initialized MLflow to track repo "sentongo-web/Kidney_classification_Using_MLOPS_and_DVC_Data-version-control"

Repository sentongo-web/Kidney_classification_Using_MLOPS_and_DVC_Data-version-control initialized!

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [4]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [5]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:
        config = self.config.evaluation
        return EvaluationConfig(
            path_of_model=config.path_of_model,
            training_data=config.training_data,
            all_params=dict(config.all_params),
            mlflow_uri=config.mlflow_uri,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE,
        )

In [6]:
import tensorflow as tf
import mlflow
import mlflow.tensorflow
from urllib.parse import urlparse

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):
        datagenerator_kwargs = dict(rescale=1.0 / 255, validation_split=0.30)
        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    def log_into_mlflow(self):
        mlflow.set_tracking_uri(self.config.mlflow_uri)
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics({"loss": self.score[0], "accuracy": self.score[1]})

            if tracking_url_type_store != "file":
                mlflow.tensorflow.log_model(self.model, "model", registered_model_name="VGG16Model")
            else:
                mlflow.tensorflow.log_model(self.model, "model")

c:\Users\sento\anaconda3\envs\kidney\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(config=eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-03-01 17:37:27,235: INFO: common]: YAML file loaded successfully: config\config.yaml
[2026-03-01 17:37:27,243: INFO: common]: YAML file loaded successfully: params.yaml
[2026-03-01 17:37:27,248: INFO: common]: Created directory: artifacts
[2026-03-01 17:37:27,958: WARNING: saving_utils]: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.
Found 139 images belonging to 2 classes.
9/9 ━━━━━━━━━━━━━━━━━━━━ 26s 3s/step - accuracy: 0.5180 - loss: 11.9273
[2026-03-01 17:37:54,531: INFO: common]: JSON saved to: scores.json


2026/03/01 17:37:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/01 17:37:58 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
Successfully registered model 'VGG16Model'.
2026/03/01 17:38:38 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16Model, version 1
Created version '1' of model 'VGG16Model'.


🏃 View run awesome-ant-272 at: https://dagshub.com/sentongo-web/Kidney_classification_Using_MLOPS_and_DVC_Data-version-control.mlflow/#/experiments/0/runs/866e1b4403df4319900735e0de0a5d8a
🧪 View experiment at: https://dagshub.com/sentongo-web/Kidney_classification_Using_MLOPS_and_DVC_Data-version-control.mlflow/#/experiments/0
